# Notebook 1: Análise Exploratória de Dados (EDA) e Pré-processamento

**Unidade Curricular:** Dados e Aprendizagem Automática (DAA) — 2026/2027

**Tema:** Modelação e Predição do Perfil Etário (`AGE_RANGE`) com base no Inquérito Nacional de Saúde (INS 2019)

---

### Objetivo deste notebook

Este é o primeiro de quatro notebooks do projeto:

| Notebook | Conteúdo |
|---|---|
| **01 — EDA e Pré-processamento** | Conhecer os dados, corrigir os tipos das colunas e guardar uma versão limpa |
| 02 — Feature Engineering | Codificar as variáveis e criar novas features |
| 03 — Treino e Tuning | Treinar, validar e otimizar modelos (métrica: Macro F1) |
| 04 — Interpretabilidade e Anomalias | Explicar o modelo, detetar perfis anómalos e gerar a submissão final |

Neste notebook queremos responder a três perguntas:

1. **Como estão os dados guardados?** (tipos das colunas, códigos de falta de resposta)
2. **Que tipo de informação tem cada coluna?** (numérica, categórica, identificador, constante)
3. **Que variáveis estão mais ligadas à idade?** (para orientar a feature engineering)

## 1. Configuração do Ambiente

Importação das bibliotecas, definição dos caminhos e das constantes do projeto.

In [ ]:
import os
import warnings

from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
import seaborn as sns

# Ignoramos apenas avisos de funcionalidades que vão mudar em versões futuras do pandas
warnings.filterwarnings('ignore', category=FutureWarning)

# Configurações de visualização
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', 100)
plt.style.use('ggplot')
sns.set_palette('viridis')

# Caminhos relativos (o notebook está dentro da pasta notebooks/)
DATA_DIR = '../data'
SUBMISSIONS_DIR = '../submissions'
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

# Constantes do projeto
TARGET = 'AGE_RANGE'
RANDOM_STATE = 42
AGE_LABELS = ['15-17', '18-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49',
              '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80-84', '85+']

print("Ambiente configurado com sucesso!")
print(f"Diretório de dados: {os.path.abspath(DATA_DIR)}")
print(f"Diretório de submissões: {os.path.abspath(SUBMISSIONS_DIR)}")

## 2. Carregamento e Limpeza Inicial dos Dados

Efetuamos o carregamento dos conjuntos de treino e teste. Todas as colunas são lidas como **texto** (`dtype=str`), para que sejamos nós a decidir a conversão de cada coluna mais à frente, de forma igual no treino e no teste. Se deixássemos o pandas adivinhar os tipos, a mesma coluna podia ser lida de forma diferente nos dois ficheiros.

Aplicamos as seguintes regras de limpeza inicial:

1. Remoção de espaços em branco nos nomes das colunas e nos valores (os CSVs têm espaços de alinhamento);
2. Conversão de células vazias em `NaN`;
3. Remoção da coluna `WGT` (peso amostral do inquérito), por indicação dos docentes;
4. Conversão da variável alvo `AGE_RANGE` para inteiro (0 a 15);
5. Verificação de que treino e teste têm exatamente as mesmas colunas (exceto o alvo).

In [ ]:
def carregar(caminho):
    """Lê um CSV com todas as colunas como texto e remove espaços dos nomes e dos valores."""
    df = pd.read_csv(caminho, dtype=str, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    for c in df.columns:
        df[c] = df[c].str.strip().replace('', np.nan)
    return df


train_df = carregar(os.path.join(DATA_DIR, 'dataset_train.csv'))
test_df = carregar(os.path.join(DATA_DIR, 'dataset_test_X.csv'))

# Remoção da coluna WGT (indicação dos docentes)
train_df = train_df.drop(columns=['WGT'])
test_df = test_df.drop(columns=['WGT'])

# A variável alvo é um número de 0 a 15
train_df[TARGET] = pd.to_numeric(train_df[TARGET]).astype(int)

print(f"Dimensão do Treino: {train_df.shape}")
print(f"Dimensão do Teste: {test_df.shape}")

# Treino e teste têm de ter as mesmas colunas (o treino tem a mais apenas o alvo)
assert set(train_df.columns) - {TARGET} == set(test_df.columns), "As colunas do treino e do teste não coincidem!"
assert 'WGT' not in train_df.columns and 'WGT' not in test_df.columns, "A coluna WGT ainda está nos dados!"
print("Treino e teste têm as mesmas colunas (exceto o alvo) e a WGT foi removida.")

> **Nota:** segundo a nota metodológica do INE, os ponderadores do INS 2019 foram ajustados segundo a região NUTS II, o **grupo etário quinquenal**, o sexo, a escolaridade e a dimensão do agregado. A variável `WGT` contém, portanto, informação derivada da própria idade; usá-la seria uma fuga de informação do alvo (*data leakage*), o que justifica a sua remoção.

### 2.1 Integridade dos Dados

Antes de analisar o conteúdo, verificamos se cada linha corresponde a uma pessoa única e se os conjuntos de treino e teste estão bem separados:

- **Linhas duplicadas** (ignorando os identificadores): duplicados dariam peso a mais a certos perfis e inflacionariam a validação cruzada;
- **Identificadores de pessoa (`PID`) repetidos** dentro de cada conjunto ou partilhados entre treino e teste (fuga de informação);
- **Agregados familiares (`HHID`) partilhados**: pessoas da mesma casa têm idades relacionadas, o que pode ser informação útil.

In [ ]:
# Colunas de conteúdo (tudo exceto os identificadores)
cols_sem_id = [c for c in test_df.columns if c not in ['PID', 'HHID']]

print("--- Duplicados ---")
print(f"Linhas duplicadas no treino (sem IDs): {train_df.duplicated(subset=cols_sem_id).sum()}")
print(f"Linhas duplicadas no teste  (sem IDs): {test_df.duplicated(subset=cols_sem_id).sum()}")

print("\n--- Pessoas (PID) ---")
print(f"PID repetidos no treino: {train_df['PID'].duplicated().sum()}")
print(f"PID repetidos no teste:  {test_df['PID'].duplicated().sum()}")
print(f"PID em comum entre treino e teste: {len(set(train_df['PID']) & set(test_df['PID']))}")

print("\n--- Agregados (HHID) ---")
print(f"HHID repetidos no treino: {train_df['HHID'].duplicated().sum()}")
print(f"HHID repetidos no teste:  {test_df['HHID'].duplicated().sum()}")
print(f"HHID em comum entre treino e teste: {len(set(train_df['HHID']) & set(test_df['HHID']))}")

> **Conclusão:** não existem linhas duplicadas, pessoas (`PID`) repetidas nem agregados (`HHID`) partilhados, nem dentro de cada conjunto nem entre treino e teste. Cada linha corresponde a uma pessoa única de um agregado diferente, pelo que não há risco de fuga de informação e os identificadores não têm valor preditivo (serão removidos na limpeza final).

## 3. Análise da Variável Alvo (`AGE_RANGE`)

A variável alvo é **ordinal e multiclasse**: 16 faixas etárias, codificadas de 0 (`15-17`) a 15 (`85+`). Avaliamos a distribuição absoluta e relativa das classes para medir o nível de desequilíbrio.

Esta análise é importante por causa da métrica da competição, o **Macro F1**: o F1 é calculado para cada classe e depois é feita a média simples. Todas as classes pesam o mesmo no resultado final (1/16), independentemente do número de pessoas, por isso as classes mais pequenas são tão importantes como as maiores.

In [ ]:
contagem = train_df[TARGET].value_counts().sort_index()

dist_alvo = pd.DataFrame({
    'Faixa etária': AGE_LABELS,
    'Contagem': contagem.values,
    'Percentagem (%)': (contagem / len(train_df) * 100).round(2).values
}, index=contagem.index)
dist_alvo.index.name = TARGET

display(dist_alvo)

ratio = contagem.max() / contagem.min()
print(f"\nNº de classes: {train_df[TARGET].nunique()}")
print(f"Maior classe: {AGE_LABELS[contagem.idxmax()]} ({contagem.max()} pessoas)")
print(f"Menor classe: {AGE_LABELS[contagem.idxmin()]} ({contagem.min()} pessoas)")
print(f"Rácio de desequilíbrio (Maior/Menor): {ratio:.2f}x")

fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(x=AGE_LABELS, y=contagem.values, color='teal', ax=ax)
ax.axhline(contagem.mean(), color='black', linestyle='--', linewidth=1,
           label=f'Média por classe ({contagem.mean():.0f})')
ax.set_title('Distribuição da Variável Alvo (AGE_RANGE)')
ax.set_xlabel('Faixa Etária')
ax.set_ylabel('Frequência Absoluta')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> **Conclusão:** as classes estão desequilibradas (rácio de 6,17x entre a maior e a menor). As faixas mais jovens (`15-17`, `18-19`) são as mais pequenas: em parte porque `18-19` abrange apenas 2 anos (as restantes abrangem 5), e em parte porque o inquérito começa nos 15 anos.
>
> Como a métrica é o Macro F1, cada uma destas classes pequenas vale tanto como as maiores. Na modelação será necessário compensar o desequilíbrio (por exemplo, com `class_weight='balanced'`), caso contrário o modelo tenderá a ignorar as faixas mais raras.

### 3.1 Referências Mínimas para o Macro F1

Para avaliar se um modelo aprendeu de facto alguma coisa, precisamos de um ponto de comparação. Calculamos o Macro F1 de dois classificadores triviais, que não usam nenhuma informação das pessoas:

- **Classe maioritária:** prevê sempre a faixa mais frequente (`60-64`). Acerta nessa classe e tem F1 = 0 nas restantes 15;
- **Aleatório estratificado:** sorteia uma faixa para cada pessoa, respeitando as proporções do treino.

Qualquer modelo desenvolvido nos notebooks seguintes tem de ficar claramente acima destes valores.

In [ ]:
y = train_df[TARGET].values
rng = np.random.default_rng(RANDOM_STATE)

# 1) Prevê sempre a classe mais frequente
pred_maioritaria = np.full(len(y), contagem.idxmax())

# 2) Sorteia uma classe para cada pessoa, com as proporções do treino
proporcoes = (contagem / contagem.sum()).values
pred_aleatoria = rng.choice(contagem.index, size=len(y), p=proporcoes)

f1_maj = f1_score(y, pred_maioritaria, average='macro', zero_division=0)
f1_rand = f1_score(y, pred_aleatoria, average='macro', zero_division=0)

print(f"Macro F1 — Classe maioritária ({AGE_LABELS[contagem.idxmax()]}): {f1_maj:.4f}")
print(f"Macro F1 — Aleatório estratificado:            {f1_rand:.4f}")
print(f"Valor teórico do aleatório (1/16):             {1/16:.4f}")

> **Conclusão:** um modelo que preveja sempre a classe mais frequente obtém um Macro F1 de apenas ~0,011, e um modelo aleatório ~0,06. Isto confirma que o Macro F1 penaliza fortemente quem ignora as classes pequenas: não basta acertar nas faixas mais numerosas, é preciso distinguir todas.
>
> Estes valores são a **referência mínima** para os modelos do Notebook 3.

## 4. Códigos Especiais e Valores em Falta

Num inquérito, a ausência de informação raramente aparece como uma célula vazia (`NaN`): aparece **escrita como texto**, por exemplo "Sem resposta" ou "Não aplicável". Para o pandas estes valores são texto como outro qualquer, pelo que o `isnull()` não os deteta e daria a falsa ideia de que quase não há valores em falta.

O primeiro passo é identificar estes códigos. Para isso contamos em **quantas colunas diferentes** aparece cada valor de texto: um valor que surge em dezenas ou centenas de colunas não é uma resposta à pergunta, mas sim um código do inquérito.

> Para este inventário juntamos treino e teste **sem a variável alvo**. Não há fuga de informação: apenas observamos que valores existem em cada coluna.

In [ ]:
# Treino + teste, sem o alvo: só para ver que valores existem
features_df = pd.concat([train_df.drop(columns=TARGET), test_df], ignore_index=True)


def para_numero(s):
    """Converte texto em número (aceita vírgula decimal). O que não for número fica NaN."""
    return pd.to_numeric(s.str.replace(',', '.', regex=False), errors='coerce')


# Para cada coluna, guarda os valores de texto distintos (os que não são números)
contagem_textos = Counter()
for c in features_df.columns:
    valores = features_df[c].dropna()
    textos = valores[para_numero(valores).isna()].unique()
    contagem_textos.update(textos)

top_textos = pd.DataFrame(contagem_textos.most_common(25),
                          columns=['Valor', 'Nº de colunas onde aparece'])
display(top_textos)

> **Conclusão:** dois valores destacam-se claramente: **"Sem resposta"** (210 colunas) e **"Não aplicável"** (109 colunas). São os códigos do inquérito e têm significados diferentes:
>
> | Código | Significado | Tratamento |
> |---|---|---|
> | **"Sem resposta"** | A pergunta foi feita, mas a pessoa não respondeu | Falta de informação verdadeira |
> | **"Não aplicável"** | A pergunta **não foi feita**, por não se aplicar à pessoa | **É informação**: muitas perguntas só se fazem a certos grupos (ex.: perguntas de emprego a quem trabalha) |
>
> Os restantes valores frequentes ("Sim", "Não", "Nunca", escalas de dificuldade e frequência) são respostas verdadeiras e mantêm-se como categorias. O mesmo vale para "Não aplicável (nunca tentou ou não precisa de o fazer)", que é uma opção de resposta própria das perguntas `HA1`.

### 4.1 Peso dos Códigos Especiais em Cada Coluna

Definimos os códigos identificados e medimos a percentagem de cada um por coluna, no treino:

- **Sem resposta:** "Sem resposta" e "NR" (abreviatura usada apenas na `IN19`);
- **Não aplicável:** "Não aplicável".

Comparamos ainda com os `NaN` verdadeiros (células vazias), para mostrar a diferença entre o que o `isnull()` deteta e a realidade.

In [ ]:
COD_SEM_RESPOSTA = ['Sem resposta', 'NR']
COD_NAO_APLICAVEL = ['Não aplicável']
CODIGOS = COD_SEM_RESPOSTA + COD_NAO_APLICAVEL


def resumo_faltas(df):
    """Percentagem de NaN, 'Sem resposta' e 'Não aplicável' em cada coluna."""
    linhas = []
    for c in df.columns:
        if c == TARGET:
            continue
        s = df[c]
        linhas.append({
            'Coluna': c,
            'NaN (%)': s.isna().mean() * 100,
            'Sem resposta (%)': s.isin(COD_SEM_RESPOSTA).mean() * 100,
            'Não aplicável (%)': s.isin(COD_NAO_APLICAVEL).mean() * 100,
        })
    return pd.DataFrame(linhas).set_index('Coluna').round(2)


faltas_treino = resumo_faltas(train_df)

print("--- O que o isnull() vê vs. a realidade (treino) ---")
print(f"Colunas com NaN verdadeiros:   {(faltas_treino['NaN (%)'] > 0).sum()} -> {list(faltas_treino.index[faltas_treino['NaN (%)'] > 0])}")
print(f"Colunas com 'Sem resposta':    {(faltas_treino['Sem resposta (%)'] > 0).sum()}")
print(f"Colunas com 'Não aplicável':   {(faltas_treino['Não aplicável (%)'] > 0).sum()}")

print("\nTop 10 colunas com mais 'Sem resposta' (treino):")
display(faltas_treino.sort_values('Sem resposta (%)', ascending=False).head(10))

print("\nTop 10 colunas com mais 'Não aplicável' (treino):")
display(faltas_treino.sort_values('Não aplicável (%)', ascending=False).head(10))

> **Conclusão:** o `isnull()` deteta apenas 1 coluna com valores em falta (`IN16`), mas na realidade:
>
> - **"Sem resposta"** surge em 213 colunas, sempre com percentagens baixas (máximo de 6,6% na `IN19`). As pessoas responderam a praticamente todas as perguntas que lhes foram feitas;
> - **"Não aplicável"** surge em 109 colunas e domina em várias delas (até 99,9% na `IN13`). São perguntas de seguimento, feitas apenas a uma pequena parte dos inquiridos.
>
> Colunas com mais de 99% de "Não aplicável" são praticamente constantes e terão pouco valor preditivo. A decisão de as manter será tomada no Notebook 2. Na secção 8 verificamos se a aplicabilidade destas perguntas depende da idade.

### 4.2 O Caso da Coluna `IN16`

A `IN16` (motivo da última ida ao dentista) é a única coluna com valores `NaN` verdadeiros (~50% das linhas). A hipótese é que esta pergunta só foi feita a quem foi ao dentista **nos últimos 12 meses**, informação registada na coluna `AM1`. Se a hipótese se confirmar, os `NaN` são um "Não aplicável" mal codificado e devem ser tratados como tal.

Verificamos cruzando o preenchimento da `IN16` com as respostas da `AM1`.

In [ ]:
# Cruzamento: respostas da AM1 vs. IN16 vazia/preenchida
tabela_in16 = pd.crosstab(train_df['AM1'],
                          train_df['IN16'].isna().map({True: 'IN16 vazia (NaN)', False: 'IN16 preenchida'}))
display(tabela_in16)

# Hipótese: IN16 está vazia exatamente quando a pessoa NÃO foi ao dentista nos últimos 12 meses
foi_dentista_12m = train_df['AM1'].isin(['Há menos de 6 meses', '6 a menos de 12 meses'])
hipotese_ok = (train_df['IN16'].isna() == ~foi_dentista_12m).all()

print(f"A IN16 está vazia exatamente para quem não foi ao dentista nos últimos 12 meses? {hipotese_ok}")

> **Conclusão:** a hipótese confirma-se sem exceções. A `IN16` só está preenchida para quem foi ao dentista nos últimos 12 meses (`AM1`); para todos os outros (5 602 pessoas no treino) a pergunta foi saltada pelo guião do inquérito.
>
> Os `NaN` da `IN16` são, portanto, um **"Não aplicável" mal codificado** e serão convertidos para "Não aplicável" na secção 6. Não devem ser imputados com outra resposta, o que criaria informação falsa (motivos de consulta para quem não foi ao dentista).

## 5. Tipificação das Colunas

Antes de preparar os dados é necessário perceber **o tipo de cada variável** (numérica, categórica, ordinal). Neste dataset todas as colunas foram lidas como texto, mas muitas contêm números (alturas, pesos, idades, contagens).

O problema está nas colunas que **misturam números com códigos**: por exemplo, a `BM1` (altura) tem valores como "165" e "Sem resposta", pelo que o pandas a trata como texto. Se estas colunas fossem tratadas como categóricas, cada número tornar-se-ia uma categoria independente, perdendo-se a ordem (180 > 158).

Classificamos cada coluna depois de **retirar os códigos especiais** identificados na secção 4:

| Tipo | Regra |
|---|---|
| **Identificador** | `PID`, `HHID` |
| **Constante** | Um único valor em todas as linhas (não tem informação e será removida) |
| **Numérica** | Apenas números, sem códigos |
| **Numérica mista** | ≥ 90% de números depois de retirar os códigos |
| **Categórica** | As restantes |

In [ ]:
COLS_ID = ['PID', 'HHID']
COLS_CONSTANTES = [c for c in features_df.columns if features_df[c].nunique(dropna=False) <= 1]


def tipo_coluna(s, limiar=0.9):
    """Classifica uma coluna depois de retirar os códigos especiais."""
    valores = s.dropna()
    valores = valores[~valores.isin(CODIGOS)]          # retira "Sem resposta", "NR", "Não aplicável"
    if len(valores) == 0:
        return 'categórica', 0.0, []
    numeros = para_numero(valores)
    pct_num = numeros.notna().mean()                   # % de valores que são números
    outros_textos = sorted(valores[numeros.isna()].unique().tolist())
    if pct_num == 1 and not s.isin(CODIGOS).any():
        return 'numérica', pct_num, outros_textos
    if pct_num >= limiar:
        return 'numérica mista', pct_num, outros_textos
    return 'categórica', pct_num, outros_textos


linhas = []
for c in features_df.columns:
    if c in COLS_ID:
        tipo, pct, outros = 'identificador', np.nan, []
    elif c in COLS_CONSTANTES:
        tipo, pct, outros = 'constante', np.nan, []
    else:
        tipo, pct, outros = tipo_coluna(features_df[c])
    linhas.append({'Coluna': c,
                   'Tipo': tipo,
                   '% numérico (sem códigos)': round(pct * 100, 1),
                   'Valores únicos': features_df[c].nunique(),
                   'Outros textos': outros[:5]})

tipos_df = pd.DataFrame(linhas).set_index('Coluna')

print("Nº de colunas por tipo:")
display(tipos_df['Tipo'].value_counts().to_frame('Nº de colunas'))
print(f"Colunas constantes: {COLS_CONSTANTES}")

print("\nColunas numéricas e numéricas mistas:")
display(tipos_df[tipos_df['Tipo'].isin(['numérica', 'numérica mista'])])

> **Conclusão:** das 236 colunas, 198 são categóricas, 30 são **numéricas mistas** (números com códigos pelo meio), 4 são numéricas puras, 2 são identificadores e 2 são constantes (`PROXY`, `COUNTRY`).
>
> Nas 30 colunas mistas, depois de retirar os códigos, praticamente tudo são números. Em 7 delas sobra um valor de texto que precisa de tratamento:
> - **Limites de escala**: `">=7"` (`HHNBPERS_COD`), `">=3"` (`HHNBPERS_0_13_COD`), `">= 25"` (`HO12_COD`), `">= 60"` (`HO34_COD`), `"53 ou mais"` (`IN25`) e `"33 ou mais"` (`IN26`). Representam um número e serão convertidos nesse número;
> - **Resposta especial**: `"Gravidez sem termo (n.a.)"` na `IN24`. Não é um número, mas é informação, e será guardada à parte.
>
> Das 4 numéricas puras, duas (`REFYEAR`, `REFMONTH`) são a data da entrevista: descrevem a recolha e não a pessoa.

### 5.1 Documento-Resumo das Variáveis

Construímos uma tabela de referência com uma linha por variável, contendo:

- **Tipo**: classificação obtida na secção anterior;
- **Contexto**: tema da pergunta no questionário (as colunas estão ordenadas pelo questionário, agrupadas por tema);
- **Expectativa**: hipótese inicial sobre a relevância da variável para prever a idade (**Alta**, **Média** ou **Baixa**), definida **antes** da análise e verificada nas secções seguintes;
- **Comentários**: observações relevantes sobre a variável.

Os temas marcados como "Por identificar" ou "a confirmar" correspondem a perguntas cujo significado não é descrito na página da competição e deverá ser confirmado no questionário oficial do INS 2019.

In [297]:
# (primeira coluna do tema, contexto, expectativa) — pela ordem do questionário
TEMAS = [
    ('PID',          'Identificação e entrevista',                 'Nenhuma'),
    ('COUNTRY',      'Localização e dados pessoais',               'Baixa'),
    ('MARSTALEGAL',  'Situação conjugal',                          'Alta'),
    ('IN1',          'Migração (a confirmar)',                     'Média'),
    ('HATLEVEL',     'Escolaridade e trabalho',                    'Alta'),
    ('HHNBPERS_COD', 'Agregado familiar e rendimento',             'Alta'),
    ('IN5',          'Seguros de saúde',                           'Média'),
    ('HS1',          'Estado de saúde e doenças crónicas',         'Alta'),
    ('AC1A',         'Acidentes e faltas ao trabalho',             'Média'),
    ('PL1',          'Limitações físicas e sensoriais',            'Alta'),
    ('PC1A',         'Autonomia no dia a dia',                     'Alta'),
    ('PN1',          'Dor e saúde mental',                         'Média'),
    ('HO12_COD',     'Utilização de cuidados de saúde',            'Média'),
    ('NT1',          'Saúde reprodutiva',                          'Alta'),
    ('UN1A',         'Necessidades de saúde não satisfeitas',      'Baixa'),
    ('BM1',          'Medidas corporais',                          'Média'),
    ('PE1',          'Atividade física',                           'Média'),
    ('IN29',         'Por identificar (IN29–IN42)',                'Por avaliar'),
    ('DH1',          'Alimentação',                                'Média'),
    ('IN43',         'Tabaco',                                     'Média'),
    ('AL1',          'Álcool',                                     'Média'),
    ('IN50',         'Satisfação com a vida (escala 1–7)',         'Média'),
    ('SS1',          'Suporte social',                             'Baixa'),
    ('IC1',          'Cuidados prestados a terceiros',             'Média'),
    ('IN10',         'Deslocações (a confirmar)',                  'Média'),
    ('IN55',         'Por identificar (IN55–IN59)',                'Por avaliar'),
    ('IN67',         'Por identificar (IN67)',                     'Por avaliar'),
]
inicio_tema = {col: (contexto, expectativa) for col, contexto, expectativa in TEMAS}

# Percorre as colunas pela ordem original e atribui o tema em vigor
contexto_atual, expectativa_atual = None, None
contextos, expectativas = [], []
for c in features_df.columns:
    if c in inicio_tema:
        contexto_atual, expectativa_atual = inicio_tema[c]
    contextos.append(contexto_atual)
    expectativas.append(expectativa_atual)

tipos_df['Contexto'] = contextos
tipos_df['Expectativa'] = expectativas


def comentario(linha):
    if linha['Tipo'] in ['identificador', 'constante']:
        return 'Sem informação para o modelo: a remover'
    if linha['Tipo'] == 'numérica mista' and linha['Outros textos']:
        return f"Texto especial a tratar: {linha['Outros textos']}"
    if linha['Tipo'] == 'numérica mista':
        return 'Número com códigos especiais'
    return ''


tipos_df['Comentários'] = tipos_df.apply(comentario, axis=1)
tipos_df.loc[['REFYEAR', 'REFMONTH'], 'Comentários'] = 'Data da entrevista: não descreve a pessoa'

resumo_variaveis = tipos_df[['Tipo', 'Contexto', 'Expectativa', 'Comentários']]

# Visão por tema
resumo_temas = (resumo_variaveis
                .groupby('Contexto', sort=False)
                .agg(Nº_variáveis=('Tipo', 'size'),
                     Expectativa=('Expectativa', 'first'),
                     Tipos=('Tipo', lambda t: ', '.join(sorted(t.unique())))))
display(resumo_temas)

# Guarda o documento completo para consulta e para o relatório
resumo_variaveis.to_csv(os.path.join(DATA_DIR, 'resumo_variaveis.csv'), encoding='utf-8-sig')
print(f"Documento-resumo guardado em data/resumo_variaveis.csv ({len(resumo_variaveis)} variáveis)")

,No_variáveis,Expectativa,Tipos
Contexto,,,
Identificação e entrevista,5,Nenhuma,"constante, identificador, numérica"
Localização e dados pessoais,10,Baixa,"categórica, constante"
Situação conjugal,2,Alta,categórica
Migração (a confirmar),4,Média,"categórica, numérica mista"
Escolaridade e trabalho,6,Alta,categórica
Agregado familiar e rendimento,6,Alta,"categórica, numérica, numérica mista"
Seguros de saúde,10,Média,categórica
Estado de saúde e doenças crónicas,20,Alta,categórica
Acidentes e faltas ao trabalho,8,Média,categórica


Documento-resumo guardado em data/resumo_variaveis.csv (236 variáveis)


> **Conclusão:** as 236 variáveis distribuem-se por 27 temas, pela ordem do questionário. Os temas com mais variáveis são a **utilização de cuidados de saúde** (31), o **estado de saúde e doenças crónicas** (20), a **autonomia no dia a dia** (17) e o **tabaco** (16). Os temas de saúde, limitações físicas e autonomia são aqueles em que esperamos maior relação com a idade.

### 5.2 Casos Especiais

#### a) Limites de escala

Algumas colunas numéricas agrupam os valores mais altos numa única categoria de texto (ex.: `">=7"` = 7 ou mais pessoas). Verificamos qual o número associado a cada limite, que será usado na conversão.

In [ ]:
def valor_limite(texto):
    """'>=7' -> 7 ; '53 ou mais' -> 53 ; outro texto -> None"""
    numero = texto.replace('>=', '').replace('ou mais', '').strip()
    return int(numero) if numero.isdigit() else None

mistas = tipos_df[tipos_df['Tipo'] == 'numérica mista']

limites = []
for col, textos in mistas['Outros textos'].items():
    for t in textos:
        limites.append({'Coluna': col,
                        'Texto': t,
                        'Valor numérico': valor_limite(t),
                        'Nº ocorrências': (features_df[col] == t).sum(),
                        'Máximo dos restantes': para_numero(features_df[col]).max()})

limites_df = pd.DataFrame(limites)
display(limites_df)

> **Conclusão:** existem 6 colunas com limite de escala (`HHNBPERS_COD`, `HHNBPERS_0_13_COD`, `HO12_COD`, `HO34_COD`, `IN25`, `IN26`). Em todas, o valor do limite é superior ao máximo dos restantes valores, pelo que substituir o texto pelo número (ex.: `">=7"` → 7) preserva a ordem. A `IN24` não tem um limite de escala: os seus valores são **anos** (1969–2019), provavelmente o da última gravidez, e a resposta especial "Gravidez sem termo" é tratada como estado. Na conversão passa a anos decorridos até à entrevista (`REFYEAR - IN24`).

#### b) Durações (`PE7`, `PE9`)

As colunas de duração da atividade física parecem estar no formato **HHMM** (ex.: `130` = 1h30). Para o confirmar, verificamos os dois últimos dígitos: se representarem minutos, nunca devem ser 60 ou mais.

In [ ]:
for c in ['PE7', 'PE9']:
    v = para_numero(features_df[c]).dropna()
    minutos = v % 100          # dois últimos dígitos
    print(f"{c}: mín = {v.min():.0f} | máx = {v.max():.0f} | "
          f"valores com 'minutos' >= 60: {(minutos >= 60).sum()} de {len(v)}")
    print("   valores mais frequentes:", v.value_counts().head(8).index.astype(int).tolist())

> **Conclusão:** em nenhum valor de `PE7` ou `PE9` os dois últimos dígitos são iguais ou superiores a 60, e os valores mais frequentes correspondem a horas exatas e meias horas (`100`, `130`, `200`). Confirma-se que estas colunas estão no formato **HHMM**, pelo que serão convertidas para minutos (`horas × 60 + minutos`): por exemplo, `130` → 90 minutos.
>
> A `PE9` está preenchida para quase todos os inquiridos e varia entre 0 e 24 horas (provavelmente tempo diário passado sentado, a confirmar); a `PE7` só está preenchida para 3844 pessoas, sendo "Não aplicável" para quem não pratica desporto.

#### c) Data da última vacina da gripe (`PA1`)

A coluna PA1 (última vacina da gripe) deveria conter uma data. Verificamos os formatos existentes pelo número de caracteres. 

In [ ]:
pa1 = features_df['PA1'].dropna()
pa1 = pa1[~pa1.isin(CODIGOS)]

print("Nº de caracteres:")
print(pa1.str.len().value_counts())
print("\nExemplos com 6 caracteres:", pa1[pa1.str.len() == 6].unique()[:8].tolist())
print("Exemplos com 4 caracteres:", pa1[pa1.str.len() == 4].unique()[:8].tolist())
print("\nCódigos na PA1:")
print(features_df['PA1'][features_df['PA1'].isin(CODIGOS)].value_counts())

print(pa1[pa1.str.len().isin([5, 16])].value_counts())

> **Conclusão:** a `PA1` não é uma coluna de datas, mas uma resposta com cinco situações possíveis:
> - **`Nunca`** (8269): nunca foi vacinado;
> - **`Há mais de 1 ano`** (2941): a última vacina foi há mais de um ano;
> - **data `AAAAMM`** (3201): vacinado no último ano, com mês indicado;
> - **apenas o ano `2018`** (67): vacinado, mas sem mês;
> - **`Sem resposta`** (139).
>
> Por ser maioritariamente texto (~78%), foi classificada como categórica. Na conversão será dividida em duas colunas: `PA1_ESTADO`, com a situação ("Nunca", "Há mais de 1 ano", "Data", "Só ano", "Sem resposta"), e `PA1_MESES`, com o número de meses entre a vacinação e a entrevista (calculado com `REFYEAR`/`REFMONTH`), preenchida apenas quando existe data completa. Espera-se que esta variável esteja fortemente associada à idade, já que a vacina da gripe é recomendada a partir dos 65 anos.

## 6. Conversão de Tipos

Cada coluna numérica mista é dividida em **valor** (número, ou NaN) e **`_ESTADO`** (`"Valor"` ou o código original), para que os códigos não sejam confundidos com números (ex.: `"Não aplicável"` não passa a 0). Além disso: a `IN24` (um ano, provavelmente o da última gravidez) passa a anos decorridos até à entrevista, os limites de escala passam a número (`">=7"` → 7), `PE7`/`PE9` passam de HHMM para minutos, a `PA1` divide-se em `PA1_ESTADO` e `PA1_MESES` (meses desde a vacina até à entrevista) e os NaN da `IN16` passam a `"Não aplicável"`. A mesma função é aplicada ao treino e ao teste.

In [ ]:
COLS_NUMERICAS = tipos_df.index[tipos_df['Tipo'] == 'numérica'].tolist()
COLS_MISTAS = tipos_df.index[tipos_df['Tipo'] == 'numérica mista'].tolist()
LIMITES = {t: str(valor_limite(t)) for t in limites_df['Texto'] if valor_limite(t) is not None}


def converter_tipos(df):
    df = df.copy()

    # Numéricas puras
    for c in COLS_NUMERICAS:
        df[c] = para_numero(df[c])

    # Numéricas mistas -> valor + _ESTADO
    for c in COLS_MISTAS:
        valor = para_numero(df[c].replace(LIMITES))
        df[c + '_ESTADO'] = np.where(valor.notna(), 'Valor', df[c])
        df[c] = valor

    # HHMM -> minutos
    for c in ['PE7', 'PE9']:
        df[c] = (df[c] // 100) * 60 + df[c] % 100

    # IN24 é um ano -> anos decorridos até à entrevista
    df['IN24'] = df['REFYEAR'] - df['IN24']

    # PA1 -> estado + meses desde a vacina
    data_completa = (df['PA1'].str.len() == 6) & df['PA1'].str.isdigit()
    so_ano = (df['PA1'].str.len() == 4) & df['PA1'].str.isdigit()
    df['PA1_ESTADO'] = df['PA1'].mask(data_completa, 'Data').mask(so_ano, 'Só ano')
    ano = pd.to_numeric(df['PA1'].where(data_completa).str[:4])
    mes = pd.to_numeric(df['PA1'].where(data_completa).str[4:])
    df['PA1_MESES'] = (df['REFYEAR'] - ano) * 12 + (df['REFMONTH'] - mes)
    df = df.drop(columns='PA1')

    # IN16: NaN = pergunta não aplicável
    df['IN16'] = df['IN16'].fillna('Não aplicável')
    return df


train_proc = converter_tipos(train_df)
test_proc = converter_tipos(test_df)
print(f"Treino: {train_df.shape} -> {train_proc.shape} | Teste: {test_df.shape} -> {test_proc.shape}")

cols_texto = train_proc.select_dtypes(include='object').columns
print("NaN em colunas de texto:", train_proc[cols_texto].isna().sum().sum())
display(train_proc[['HHNBPERS_COD', 'BM1', 'BM2', 'PE7', 'PE9', 'PA1_MESES', 'IN24']].describe().round(1))

> **Conclusão:** após a conversão, o treino passou a ter 268 colunas (30 colunas `_ESTADO`, `PA1_ESTADO` e `PA1_MESES`, sem a `PA1`). Não há NaN nas colunas de texto, e os valores numéricos estão dentro de intervalos plausíveis: altura entre 130 e 205 cm, peso entre 32 e 150 kg, até 30 h semanais de desporto e 24 h diárias na `PE9`, vacinação da gripe entre 0 e 15 meses antes da entrevista e anos desde a última gravidez entre 0 e 50.
> As colunas `IN55`–`IN59` têm valores elevados (até 8000), mas não são anos: são valores redondos (5, 10, 50, 100), provavelmente montantes, distâncias ou tempos (unidade a confirmar), e mantêm-se como estão. Os valores extremos são analisados na secção seguinte.

## 7. Análise Univariada das Variáveis Numéricas

Analisamos a distribuição de cada variável numérica através de estatísticas descritivas, da **assimetria** (*skewness*: valores acima de 1 indicam uma cauda longa à direita, ou seja, poucos valores muito altos) e da **curtose** (*kurtosis*: valores elevados indicam a presença de valores extremos), complementadas com histogramas.

In [ ]:
cols_valor = [c for c in COLS_NUMERICAS + COLS_MISTAS if c not in ['REFYEAR', 'REFMONTH']] + ['PA1_MESES']

estat = train_proc[cols_valor].describe().T[['count', 'mean', '50%', 'min', 'max']]
estat['skew'] = train_proc[cols_valor].skew()
estat['kurtosis'] = train_proc[cols_valor].kurt()
display(estat.round(1).sort_values('skew', ascending=False))

train_proc[cols_valor].hist(bins=30, figsize=(18, 14), color='teal')
plt.suptitle('Distribuição das variáveis numéricas (Treino)', y=1.0)
plt.tight_layout()
plt.show()

> **Conclusão:** a maioria das variáveis numéricas **não segue uma distribuição normal**:
> - **Distribuição aproximadamente simétrica:** apenas `BM1` (altura), `BM2` (peso) e `IN45` (skew < 0,7);
> - **Contagens concentradas em 0** (`HO12_COD`, `HO34_COD`, `AM3`, `AM5`, `PE4`, `PE8`): a maioria das pessoas não teve o evento;
> - **Valores extremos acentuados** em `IN55`–`IN59` (skew até 24, curtose até 622): poucas pessoas com valores 50 a 100 vezes acima da mediana;
> - **`PA1_MESES` bimodal:** vacinação na campanha de outono de 2019 (0–2 meses) ou na de 2018 (~11 meses).
>
> Algumas variáveis parecem representar **idades** (IN47 e IN45 no bloco do tabaco, possivelmente idade de início e de abandono, respetivamente; `IN2`–`IN4` no bloco da migração). Se se confirmar, são especialmente úteis, já que a idade atual é sempre igual ou superior a esses valores. A `SK2` tem um pico em 20, consistente com o número de cigarros por dia (um maço).
>
> Não aplicamos transformações nesta fase: a assimetria e os valores extremos serão tratados no Notebook 2, conforme o modelo escolhido (os modelos baseados em árvores não são sensíveis a estas escalas).

## 8. Relação das Variáveis com a Idade

### 8.1 Variáveis Numéricas

Medimos a relação de cada variável numérica com o alvo através da **correlação de Spearman**, adequada porque o alvo é ordinal e muitas variáveis são assimétricas (o Spearman usa a ordem dos valores, sendo robusto a valores extremos). A correlação é calculada apenas com as pessoas que têm valor em cada variável. As variáveis mais correlacionadas são depois visualizadas com boxplots por faixa etária.

In [ ]:
corr_alvo = train_proc[cols_valor + [TARGET]].corr(method='spearman')[TARGET].drop(TARGET)

corr_df = pd.DataFrame({'Spearman': corr_alvo.round(2),
                        'Nº com valor': train_proc[cols_valor].notna().sum()})
corr_df = corr_df.sort_values('Spearman', key=abs, ascending=False)
display(corr_df)

# Boxplots das 8 variáveis mais correlacionadas (em valor absoluto)
top8 = corr_df.index[:8]
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, c in zip(axes.flat, top8):
    sns.boxplot(data=train_proc, x=TARGET, y=c, color='teal', showfliers=False, ax=ax)
    ax.set_title(f"{c} (Spearman = {corr_df.loc[c, 'Spearman']})")
    ax.set_xticks(range(16))
    ax.set_xticklabels(AGE_LABELS, rotation=90)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 11))
sns.heatmap(train_proc[cols_valor].corr(method='spearman'), cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3)
plt.title('Correlação de Spearman entre variáveis numéricas (Treino)')
plt.tight_layout()
plt.show()

> **Conclusão:** as variáveis numéricas mais relacionadas com a idade são:
> - **`IN24`** (anos desde a última gravidez, ρ = 0,81): só existe para mulheres até aos 59 anos, pelo que a própria ausência de resposta é informativa;
> - **`HH_ACT`** (ρ = −0,66): é sempre 0 a partir dos 65 anos, o que indica que conta as pessoas em idade ativa no agregado; o mesmo explica a correlação negativa da `HH_INACT`;
> - **Composição do agregado** (`HHNBPERS_COD`, `HHNBPERS_0_13_COD`): agregados maiores e com crianças nas idades jovens e intermédias. No caso das crianças a relação não é monótona (irmãos aos 15-19 anos, filhos aos 30-49), pelo que o Spearman a subestima;
> - **`IN45`** (ρ = 0,43): cresce com a idade e está sempre abaixo da idade atual, o que é consistente com a idade em que deixou de fumar. A `IN47` (ρ = 0,02) não varia com a idade, o que é consistente com a idade de início do consumo de tabaco (mediana ~17 anos);
> - **`IN2`–`IN4`** (bloco da migração): crescem com a idade (significado a confirmar);
> - **`BM1`** (altura, ρ = −0,30) e dias de exercício (`PE6`, `PE8`): os mais velhos são em média mais baixos e praticam menos exercício.
>
> O peso (`BM2`, ρ = 0,00) e os meses desde a vacina (`PA1_MESES`) não têm relação monótona com a idade. Como o Spearman só deteta relações monótonas, variáveis como estas podem ainda ser úteis a modelos baseados em árvores.
>
> Entre variáveis, nenhuma correlação ultrapassa ~0,7, pelo que não há redundâncias evidentes a eliminar.
>
> Foi identificado um caso incoerente: uma pessoa da faixa 15-17 anos com ~16 anos desde a última gravidez. Casos deste tipo (valores incompatíveis com a faixa etária) serão analisados no Notebook 4, na deteção de anomalias.

### 8.2 Variáveis Categóricas

Para cada variável categórica (incluindo as colunas `_ESTADO`), calculamos a faixa etária média de cada resposta. A **amplitude** (diferença entre a resposta com faixa média mais alta e a mais baixa) mede o quanto a resposta varia com a idade: 1 unidade corresponde a uma faixa (~5 anos). Consideramos apenas respostas com pelo menos 30 pessoas. Comparamos ainda os resultados por tema com as expectativas definidas na secção 5.1.

In [ ]:
cols_cat = [c for c in train_proc.select_dtypes(include='object').columns
            if c not in COLS_ID + COLS_CONSTANTES]

linhas = []
for c in cols_cat:
    grupos = train_proc.groupby(c)[TARGET].agg(['mean', 'size'])
    grupos = grupos[grupos['size'] >= 30]
    if len(grupos) < 2:
        continue
    linhas.append({'Coluna': c,
                   'Amplitude': round(grupos['mean'].max() - grupos['mean'].min(), 1),
                   'Resposta mais jovem': grupos['mean'].idxmin(),
                   'Resposta mais velha': grupos['mean'].idxmax(),
                   'Contexto': tipos_df.loc[c.replace('_ESTADO', ''), 'Contexto']})

assoc_cat = pd.DataFrame(linhas).set_index('Coluna').sort_values('Amplitude', ascending=False)
print(f"Variáveis analisadas: {len(assoc_cat)}")
display(assoc_cat.head(20))

# Confronto com as expectativas da secção 5.1
por_tema = assoc_cat.groupby('Contexto')['Amplitude'].agg(['max', 'mean']).round(1)
por_tema['Expectativa'] = tipos_df.groupby('Contexto')['Expectativa'].first()
display(por_tema.sort_values('max', ascending=False))

idade = train_proc[TARGET].map(dict(enumerate(AGE_LABELS)))
display(pd.crosstab(idade, train_proc['HA1A'] == 'Não aplicável').rename(columns={True: 'Não aplicável', False: 'Respondeu'}))

top4 = assoc_cat.index[:4]
fig, axes = plt.subplots(2, 2, figsize=(20, 10))
for ax, c in zip(axes.flat, top4):
    tabela = pd.crosstab(train_proc[c], train_proc[TARGET], normalize='columns') * 100
    sns.heatmap(tabela, cmap='viridis', ax=ax, cbar_kws={'label': '%'})
    ax.set_title(f"{c} (amplitude = {assoc_cat.loc[c, 'Amplitude']})")
    ax.set_xticklabels(AGE_LABELS, rotation=90)
    ax.set_xlabel('')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

> **Conclusão:** a variável categórica mais relacionada com a idade é a **situação profissional** (`MAINSTAT`, amplitude 11,2 faixas): estudantes até aos 20-24 anos, empregados entre os 25 e os 59, reformados a partir dos 65. Seguem-se o **estado civil** (`MARSTALEGAL`, de solteiro a viúvo), a **escolaridade** (`HATLEVEL`) e o **estado de saúde autoavaliado** (`HS1`).
>
> A descoberta principal está nas 13 perguntas de **autonomia no dia a dia** (`HA1*`, `PC1*`): só são feitas a pessoas com **55 ou mais anos**, sendo "Não aplicável" para todos os restantes. O próprio desenho do questionário separa, assim, as faixas abaixo e acima dos 55 anos. Como estas colunas partilham o mesmo padrão, são redundantes entre si e poderão ser resumidas no Notebook 2.
>
> Por tema, as expectativas da secção 5.1 confirmam-se na maioria dos casos (trabalho, autonomia, situação conjugal, saúde). As exceções foram as **medidas corporais** e os **cuidados prestados a terceiros**, quase sem relação com a idade, e as **limitações físicas**, abaixo do esperado.
>
> A verificação na `HA1A` confirma que o corte é **perfeito**: todas as pessoas até aos 50-54 anos têm "Não aplicável" e todas a partir dos 55-59 anos responderam. Esta coluna, sozinha, separa sem erro as 9 faixas mais jovens das 7 mais velhas.

## 9. Remoção de Colunas e Gravação dos Dados

Removemos as colunas sem informação sobre a pessoa: o identificador do agregado (`HHID`), as constantes (`PROXY`, `COUNTRY`) e a data da entrevista (`REFYEAR`, `REFMONTH`), já usada nas conversões. A `NUTS1` é removida se for totalmente determinada pela `NUTS2_AG`. O `PID` passa a índice, para identificar as pessoas no Notebook 4.

Os dados finais são guardados em CSV, com o `PID` como índice, mantendo a ordem original das linhas do teste (necessária para a submissão).

In [ ]:
COLS_REMOVER = ['HHID', 'PROXY', 'COUNTRY', 'REFYEAR', 'REFMONTH']

# NUTS1 é redundante se cada NUTS2_AG pertencer a uma única NUTS1
if train_proc.groupby('NUTS2_AG')['NUTS1'].nunique().max() == 1:
    COLS_REMOVER.append('NUTS1')
print("Colunas removidas:", COLS_REMOVER)

train_final = train_proc.drop(columns=COLS_REMOVER).set_index('PID')
test_final = test_proc.drop(columns=COLS_REMOVER).set_index('PID')

# O teste tem de ter as mesmas colunas, pela mesma ordem (sem o alvo)
assert list(train_final.columns.drop(TARGET)) == list(test_final.columns)

train_final.to_csv(os.path.join(DATA_DIR, 'train_nb01.csv'))
test_final.to_csv(os.path.join(DATA_DIR, 'test_nb01.csv'))

print(f"Treino final: {train_final.shape} | Teste final: {test_final.shape}")
print(f"Numéricas: {len(train_final.select_dtypes(include='number').columns) - 1} | "
      f"Categóricas: {len(train_final.select_dtypes(include='object').columns)}")

## 10. Conclusões

**Qualidade dos dados.** Não existem duplicados nem pessoas ou agregados repetidos. A falta de informação está codificada como texto: "Sem resposta" é rara (máx. 6,6% por coluna), enquanto "Não aplicável" é frequente e informativa, por corresponder a perguntas que o questionário só faz a certos grupos. Os `NaN` da `IN16` são um "Não aplicável" mal codificado.

**Pré-processamento.** As 30 colunas numéricas mistas foram divididas em valor e estado (`_ESTADO`); os limites de escala foram convertidos em números, as durações HHMM em minutos, a `PA1` em estado e meses desde a vacina, e a `IN24` (ano) em anos decorridos. Foram removidos identificadores, constantes, a data da entrevista e a `NUTS1` (redundante). O peso amostral `WGT` foi excluído por ter sido calibrado com a própria idade.

**Relação com a idade.** As variáveis mais informativas são:
- o **desenho do questionário**: as perguntas de autonomia (`HA1*`, `PC1*`) só são feitas a quem tem 55 ou mais anos, separando sem erro as 9 faixas mais jovens das 7 mais velhas;
- a **situação profissional** (`MAINSTAT`), o **estado civil** (`MARSTALEGAL`) e a **escolaridade** (`HATLEVEL`);
- a **composição do agregado** (`HH_ACT`, `HHNBPERS_COD`, crianças dos 0 aos 13 anos);
- variáveis com natureza de idade ou tempo decorrido: anos desde a última gravidez (`IN24`), idade de abandono do tabaco (`IN45`);
- o **estado de saúde** autoavaliado (`HS1`), que piora com a idade.

**Perfil preliminar das faixas etárias:**
| Faixas | Perfil típico |
|---|---|
| 15-24 | Estudantes, solteiros, a viver em agregados grandes (com pais e irmãos) |
| 25-54 | Empregados, casados, com filhos menores no agregado; mulheres com gravidez recente |
| 55-64 | Transição para a reforma; passam a responder às perguntas de autonomia, quase sempre sem dificuldades |
| 65+ | Reformados, viúvos com frequência crescente, agregados de 1-2 pessoas, pior saúde autoavaliada e dificuldades crescentes no dia a dia |

**Desafios para a modelação.** O desequilíbrio das classes (rácio 6,2x) e a métrica Macro F1 exigem atenção às faixas pequenas (`15-17`, `18-19`). A distinção entre faixas vizinhas dentro do mesmo grupo de vida (ex.: `35-39` vs `40-44`) será a parte mais difícil, por existirem poucas variáveis que mudem de forma marcada nesses intervalos.

**Próximo passo (Notebook 2):** codificação das variáveis categóricas (incluindo as escalas ordinais), tratamento dos valores em falta das numéricas e criação de novas features (ex.: IMC, número de dificuldades no dia a dia, indicadores de "Não aplicável").